In [1]:
# Install Surprise
!pip install scikit-surprise

  Using cached scikit_surprise-1.1.4.tar.gz (154 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp310-cp310-linux_x86_64.whl size=2357282 sha256=946cf53b26497bf997ecb314a83c25d219ac8ad9960a164627ba2c058d887244
  Stored in directory: /root/.cache/pip/wheels/4b/3f/df/6acbf0a40397d9bf3ff97f582cc22fb9ce66adde75bc71fd54
Successfully built scikit-surprise


In [2]:
# Import necessary libraries
import pandas as pd
from surprise import Dataset, Reader, KNNBasic
from surprise import accuracy
from google.colab import drive

In [4]:
# Mount Google Drive
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/'  # Update this path

# Load datasets
train_sessions = pd.read_csv(drive_path + 'train_sessions.csv')
train_purchases = pd.read_csv(drive_path + 'train_purchases.csv')
item_features = pd.read_csv(drive_path + 'item_features.csv')
candidate_items = pd.read_csv(drive_path + 'candidate_items.csv')
test_sessions = pd.read_csv(drive_path + 'test_leaderboard_sessions.csv')  # Use desired test set
test_purchases = pd.read_csv(drive_path + 'test_leaderboard_purchases.csv')  # Correct test set purchases

Mounted at /content/drive


In [5]:
# Prepare training data for Surprise
# Merge session data with purchases
train_data = train_sessions.merge(train_purchases, on='session_id', how='left')

# For Surprise, we need session_id, item_id, and a rating column
train_data['rating'] = 1  # Implicit feedback, all interactions have equal weight
train_data = train_data[['session_id', 'item_id', 'rating']]

# Filter for candidate items in training data
train_data = train_data[train_data['item_id'].isin(candidate_items['item_id'])]

# Prepare test data
test_data = test_sessions[['session_id', 'item_id']].copy()

KeyError: "['item_id'] not in index"

In [6]:
# Merge training sessions with purchases
train_data = train_sessions.merge(
    train_purchases,
    on='session_id',
    how='left',
    suffixes=('_viewed', '_purchased')  # Avoid duplicate column names
)

# Ensure correct item column is used
train_data['item_id'] = train_data['item_id_viewed']  # Use viewed items as interactions
train_data = train_data[['session_id', 'item_id']]  # Keep only session and item columns

# Add a rating column for implicit feedback
train_data['rating'] = 1  # Assign a rating of 1 for each interaction

# Filter for candidate items
train_data = train_data[train_data['item_id'].isin(candidate_items['item_id'])]

/tmp/ipykernel_2602/4218013618.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data['rating'] = 1  # Assign a rating of 1 for each interaction


In [7]:
# Load test data
test_sessions = pd.read_csv(drive_path + 'test_leaderboard_sessions.csv')  # Use desired test set
test_purchases = pd.read_csv(drive_path + 'test_leaderboard_purchases.csv')

In [8]:
# Prepare Surprise dataset
reader = Reader(rating_scale=(1, 1))  # Rating scale for implicit feedback
surprise_data = Dataset.load_from_df(train_data[['session_id', 'item_id', 'rating']], reader)


In [9]:
# Build trainset for Surprise
trainset = surprise_data.build_full_trainset()

In [10]:
# Train model with KNN
from surprise import KNNBasic

algo = KNNBasic(sim_options={'name': 'cosine', 'user_based': False})  # Item-based collaborative filtering
algo.fit(trainset)


Computing the cosine similarity matrix...
Done computing similarity matrix.


In [11]:
# Generate recommendations for test sessions
test_sessions = test_sessions[['session_id', 'item_id']]  # Ensure we use the correct columns


In [12]:
# Define a function to get top-N recommendations
def get_top_n_recommendations(session_data, algo, n=10):
    recommendations = {}
    for session_id, group in session_data.groupby('session_id'):
        viewed_items = group['item_id'].unique()
        scores = []
        for item in candidate_items['item_id']:
            if item not in viewed_items:  # Exclude already viewed items
                score = algo.predict(session_id, item).est
                scores.append((item, score))
        scores.sort(key=lambda x: x[1], reverse=True)  # Sort by score
        recommendations[session_id] = [item for item, _ in scores[:n]]
    return recommendations


In [15]:
# Get top-10 recommendations
top_n = get_top_n_recommendations(test_sessions, algo, n=10)


In [16]:
# Save recommendations
submission = []
for session_id, items in top_n.items():
    for rank, item_id in enumerate(items):
        submission.append([session_id, item_id, rank + 1])

submission_df = pd.DataFrame(submission, columns=['session_id', 'item_id', 'rank'])
submission_df.sort_values(by=['session_id', 'rank'], inplace=True)


In [22]:
# Save to Google Drive
submission_df.to_csv(drive_path + 'submission.csv', index=False)
print("Recommendations saved to Google Drive.")

Recommendations saved to Google Drive.
